In [1]:
%pip install -q groq

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ['GROQ_API_KEY'] = "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" # Sendo literal ao video como requisitado

In [3]:
import os
from groq import Groq

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

chat_completion = client.chat.completions.create(
    messages=[
        {"role": "user", "content": "Explain the importance of fast language models"}
    ],
    model="llama-3.3-70b-versatile", # 'llama3-8b-8192' foi descontinuado
    temperature=0 # Acaba com a criatividade do modelo
)

print(chat_completion.choices[0].message.content)

Fast language models are crucial in today's technology landscape, and their importance can be understood from several perspectives:

1. **Efficient Processing**: Fast language models can process and analyze large amounts of text data quickly, making them ideal for applications where speed is essential, such as real-time chatbots, virtual assistants, and sentiment analysis.
2. **Improved User Experience**: Fast language models enable faster response times, which leads to a better user experience. Users can interact with language-based systems more efficiently, and the overall experience becomes more engaging and responsive.
3. **Scalability**: Fast language models can handle a large volume of requests and conversations simultaneously, making them suitable for large-scale applications, such as customer service chatbots, language translation platforms, and content generation tools.
4. **Reduced Latency**: Fast language models minimize latency, which is critical in applications where timel

In [4]:
class Agent:
    def __init__(self, client, system):
        self.client = client
        self.system = system
        self.messages = [] # 'Memória' do agente (respostas de cada iteração)
        if self.system is not None: # Se tem um texto prévio a prompt adicionar como instruções na memória
            self.messages.append({
                "role": "system", # Gravar como do sistema
                "content": self.system
            })
    
    def __call__(self, message=""):
        if message:
            self.messages.append({
                "role": "user", # Gravar como usuário (presumidamente vai poder ter uma como tool ou algo do gênero)
                "content": message
            })
        
        result = self.execute()
        self.messages.append({
                "role": "assistant",
                "content": result
            })
        return result

    def execute(self):
        completion = client.chat.completions.create(
            messages=self.messages,
            model="llama-3.3-70b-versatile", # 'llama3-8b-8192' foi descontinuado
            temperature=0
        )
        return completion.choices[0].message.content
    

In [5]:
# Um pouco de engenharia de prompt é essencial para o funcionamento de agentes desse modo, o uso de few-shot garante a estrutura, mas uma estrutura mulitagente ainda é nebulosa, presumidamente cada agente receberia sua prompt própria, mas creio que deve ter um prompt base para todas as prompts de todos agentes nesse contexto
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

get_planet_mass:
e.g. get_planet_mass: Earth
returns weight of the planet in kg

Example session:

Question: What is the mass of Earth times 2?
Thought: I need to find the mass of Earth
Action: get_planet_mass: Earth
PAUSE 

You will be called again with this:

Observation: 5.972e24

Thought: I need to multiply this by 2
Action: calculate: 5.972e24 * 2
PAUSE

You will be called again with this: 

Observation: 1,1944x10e25

If you have the answer, output it as the Answer.

Answer: The mass of Earth times 2 is 1,1944x10e25.

Now it's your turn:
""".strip()

In [6]:
# tools (extraído do git, não do vídeo)
def calculate(operation: str) -> float:
    return eval(operation)

# Será interessante ver como isso funcionaria para modelos em outros idiomas, presumidamente um modelo que recebe uma prompt em portugues também 'pensa' em português
def get_planet_mass(planet: str) -> float:
    match planet.lower():
        case "earth":
            return 5.972e24
        case "jupiter":
            return 1.898e27
        case "mars":
            return 6.39e23
        case "mercury":
            return 3.285e23
        case "neptune":
            return 1.024e26
        case "saturn":
            return 5.683e26
        case "uranus":
            return 8.681e25
        case "venus":
            return 4.867e24
        case _:
            return 0.0

In [7]:
neil_tyson = Agent(client=client, system=system_prompt)

In [8]:
result = neil_tyson("What is the mass of Earth times 5?") # 5.972e24 * 5 é quase 3e25
print(result)

Thought: To find the mass of Earth times 5, I first need to determine the mass of Earth. 

Action: get_planet_mass: Earth
PAUSE


In [9]:
neil_tyson.messages # debug

[{'role': 'system',
  'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE \n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this: \n\nObservation: 1,1944x10e25\n\nIf you have the

Uma coisa interessante aconteceu, esse modelo roda diferente do video, ele olha um passo a frente.
No vídeo, nesse momento, o assistente parou em Thought, mas aqui ele já chegou na ação e parou.

In [10]:
observation = get_planet_mass("earth")
print(observation)

5.972e+24


In [11]:
next_prompt = f"Observation {observation}"
result = neil_tyson(next_prompt)
print(result)

Thought: Now that I have the mass of Earth, I can calculate the mass of Earth times 5 by multiplying the mass of Earth by 5.

Action: calculate: 5.972e+24 * 5
PAUSE


In [12]:
neil_tyson.messages # debug
# Novamente um pensamento mais complexo e acelerado, ele naturalmente já olha para ação

[{'role': 'system',
  'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE \n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this: \n\nObservation: 1,1944x10e25\n\nIf you have the

Eu **removi** as seções com
```python
result = neil_tyson()
print(result)
```
Porque esse modelo simplesmente não precisa disso, pulando um passo de processamento e isso gerava apenas uma caixa vazia

In [13]:
observation = calculate("5.972e24 * 5")
print(observation)

2.9860000000000004e+25


In [14]:
next_prompt = f"Observation: {observation}"
result = neil_tyson(next_prompt)
print(result)

Thought: I have now calculated the mass of Earth times 5, so I can provide the answer.

Answer: The mass of Earth times 5 is 2.986e+25.


In [15]:
neil_tyson.messages

[{'role': 'system',
  'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE \n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this: \n\nObservation: 1,1944x10e25\n\nIf you have the

Agora o objetivo é implementar o **ciclo iterativo** de modo automático

In [16]:
import re
# O maior problema dessa função é o resultado de timeout, se as iterações não forem o bastante ele só desiste e vida que segue, não tem anotação, log, etc
def agent_loop(max_iterations, system, query):
    agent = Agent(client=client, system=system)
    tools = ["calculate", "get_planet_mass"]
    next_prompt = query
    i = 0
    while i < max_iterations:
        i += 1
        result = agent(next_prompt)
        print(result)

        if "PAUSE" in result and "Action" in result:
            action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
            chosen_tool = action[0][0]
            arg = action[0][1]

            if chosen_tool in tools:
                result_tool = eval(f"{chosen_tool}('{arg}')")
                next_prompt = f"Observation: {result_tool}"

            else:
                next_prompt = "Observation: Tool not found"

            print(next_prompt)
            continue

        if "Answer" in result:
            break

In [17]:
agent_loop(max_iterations=10, system=system_prompt, query="What is the mass of Earth plus the mass of Mercury, and all of that times 5?")

Thought: To solve this problem, I need to find the mass of Earth and the mass of Mercury, add them together, and then multiply the result by 5. First, I'll find the mass of Earth.

Action: get_planet_mass: Earth
PAUSE
Observation: 5.972e+24
Thought: Now that I have the mass of Earth, I need to find the mass of Mercury.

Action: get_planet_mass: Mercury
PAUSE
Observation: 3.285e+23
Thought: Now that I have the masses of both Earth and Mercury, I can add them together.

Action: calculate: 5.972e+24 + 3.285e+23
PAUSE
Observation: 6.300500000000001e+24
Thought: Now that I have the sum of the masses of Earth and Mercury, I can multiply this result by 5 to get the final answer.

Action: calculate: 6.300500000000001e+24 * 5
PAUSE
Observation: 3.1502500000000004e+25
Thought: I have now calculated the mass of Earth plus the mass of Mercury, and multiplied the result by 5, so I have the final answer.

Answer: The mass of Earth plus the mass of Mercury, and all of that times 5 is 3.15025000000000